In [2]:
import os
from dotenv import load_dotenv
from openai import OpenAI

# Load variables from the .env file
load_dotenv()

# Read the API key from environment variables
API_KEY = os.getenv("GROQ_API_KEY")  # or os.getenv("OPENAI_API_KEY")

# Initialize the OpenAI-compatible client
client = OpenAI(
    api_key=API_KEY,
    base_url="https://api.groq.com/openai/v1"  # Remove this base_url line if using OpenAI directly
)

MODEL = "llama-3.3-70b-versatile"  # Default model for Groq

print("Client ready.")

Client ready.


Part 1.1 — Your first API call

In [3]:
# TODO: Write a helper function you will reuse for the WHOLE lab:
#
# def ask_llm(user_prompt, system_prompt="You are a helpful assistant.",
#             temperature=0.7, max_tokens=500):
#     response = client.chat.completions.create(
#         model=MODEL,
#         messages=[
#             {"role": "system", "content": system_prompt},
#             {"role": "user",   "content": user_prompt},
#         ],
#         temperature=temperature,
#         max_tokens=max_tokens,
#     )
#     return response.choices[0].message.content
#
# TODO: Call it once with a simple question and print the answer.
# TODO: Print response.usage as well — how many tokens did your call consume?



def ask_llm(user_prompt, system_prompt="You are a helpful assistant.", temperature=0.7, max_tokens=500):
    
    #This is a helper function to send a prompt to the LLM and return the text response along with token usage.
    
    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user",   "content": user_prompt},
        ],
        temperature=temperature,
        max_tokens=max_tokens,
    )
    return response.choices[0].message.content, response.usage

# Test call with a simple question
prompt = "What is microfinance, and how does it help startups?"
answer, usage = ask_llm(prompt)

print("Response from LLM:")
print(answer)
print("\n Token Usage ")
print(usage)

Response from LLM:
Microfinance refers to the provision of financial services, such as loans, savings, and insurance, to individuals or small businesses that do not have access to traditional banking services. Microfinance institutions (MFIs) offer these services to low-income individuals, entrepreneurs, and small businesses, often in developing countries or underserved communities.

Microfinance helps startups in several ways:

1. **Access to capital**: Microfinance provides startups with access to small loans, which can be used to cover initial business expenses, such as purchasing equipment, renting a workspace, or hiring employees.
2. **Flexible repayment terms**: Microfinance loans often have flexible repayment terms, which can be tailored to the startup's cash flow and revenue projections.
3. **Lower interest rates**: Microfinance interest rates are often lower than those offered by traditional lenders, making it more affordable for startups to borrow money.
4. **Simplified appli

1. System role is used to set clear guidelines, context and constraints for the AI model before it processes user input eg. "You are a microfinance credit risk assessment assistant. Summarize input texts in 3 short, factual sentences without inventing details.". However, the user roles are the specific instructions, dynamic inputs given to the AI model to respond to, eg. "Summarise this loan application letter for Kwei Nii".

2. A token is a small piece of text that an AI model reads.It is usually about 4 letters or most of a word. Common words like "cane" are 1 token. Longer or rare words like "microfinance" are broken into 2 or 3 tokens.AI models use these pieces to count and charge for their work.
API providers charge per token because computational costs scale with text length.Every token requires GPU memory, processing power, and time. Processing a 50-page document uses significantly more GPU resources than a 5-word question. Billing per token ensures you only pay for the exact computational cost of your request.

Part 1.2 — Temperature: the randomness dial

In [4]:
# TODO: Ask the SAME question 5 times at temperature=0.0 and 5 times at temperature=1.2.
#   A good test question: "Suggest a name for a savings product for market traders in Accra."

# TODO: Print all 10 answers, grouped by temperature.

question = "Suggest a name for a savings product for market traders in Accra."

print(" Temperature = 0.0 (Deterministic) ")
for i in range(5):
    ans, _ = ask_llm(question, temperature=0.0)
    print(f"Run {i+1}: {ans.strip()}\n")

print("=" * 50)

print(" Temperature = 1.2 (High Randomness/Creativity) ")
for i in range(5):
    ans, _ = ask_llm(question, temperature=1.2)
    print(f"Run {i+1}: {ans.strip()}\n")

 Temperature = 0.0 (Deterministic) 
Run 1: Here are a few suggestions for a savings product for market traders in Accra:

1. **Makola Save**: "Makola" is a well-known market in Accra, so this name could resonate with market traders.
2. **Trader's Treasure**: This name emphasizes the idea of saving and accumulating wealth.
3. **Accra Amanfu**: "Amanfu" is a Ghanaian word for "savings" or "treasury", so this name incorporates local language and culture.
4. **Market Mobi**: This name is short and catchy, and "Mobi" implies mobility and flexibility, which could appeal to market traders who need to manage their finances on-the-go.
5. **Sika Su**: "Sika" is the Ghanaian word for "money", and "Su" means "grow" or "increase", so this name suggests a savings product that helps traders grow their wealth.
6. **Kokroko Savings**: "Kokroko" is a Ghanaian word for "honest" or "trustworthy", which could convey a sense of reliability and security for market traders.
7. **Traders' Trust**: This name em

1. Setting the temperature to 0.0 makes the AI model completely predictable. Because it always picks the most likely words, all 5 test runs produced nearly identical answers with strict, consistent formatting.
Setting the temperature to 1.2 the model introduced significant output variance, producing highly creative and diverse naming suggestions across the 5 iterations. However, this increased randomness makes it less structured, leading to slight verbosity and less concise phrasing.

2. A low temperature of 0.0 is best for checking loan applications. Financial tools must be accurate and consistent every single time. Lowering the temperature removes randomness, ensuring the AI gives the exact same formatting and decision for the same application.

Section 2 — The Dataset: Loan Application Letters

In [5]:
# Section 2: Load Dataset and Gold-Standard Labels

LETTERS = {
    "L001": """Dear Sir/Madam,
My name is Akosua Mensah and I have been selling provisions at Makola Market for 12 years.
I am applying for a loan of GHS 8,000 to buy a deep freezer and expand into frozen foods.
My current stall makes about GHS 900 profit each month. I have saved GHS 2,500 with your
susu scheme over the past two years and I have never missed a contribution. I can repay
GHS 450 monthly over 20 months. My sister, a teacher, will stand as my guarantor.
Thank you for considering my application.""",

    "L002": """Hello,
I am Kwame Boateng, a commercial driver in Kumasi. I need GHS 25,000 urgently to repair my
trotro engine and settle some personal debts. Business has been slow but it will surely
pick up after the festive season. I can pay back whenever the money comes. I do not have
collateral at the moment but God willing everything will be fine. Please help me quickly.""",

    "L003": """Dear Loan Committee,
I am Efua Darko, owner of Darko Fashions, a registered dressmaking business in Takoradi
(registration no. BN-2019-4482). I employ three apprentices. I request GHS 15,000 to
purchase two industrial sewing machines and fabric stock ahead of the Christmas season.
Last year my December revenue alone was GHS 22,000; monthly profit averages GHS 2,800.
I hold a fixed deposit of GHS 5,000 with GCB which I can pledge. Proposed repayment:
GHS 1,100 monthly for 15 months. Attached are my sales records for the past 18 months.""",

    "L004": """Good day,
My name is Yaw Owusu. I want a loan for my poultry farm at Nsawam. The amount is GHS 12,000
for feed and 500 new layers. I started the farm last year. Sometimes I make good money,
around GHS 1,500 in a good month, but bird flu affected us in March and I lost many birds.
I am rebuilding now. I can repay in 18 months. My uncle has agreed to guarantee the loan
with his taxi.""",

    "L005": """Dear Manager,
I am writing on behalf of the Adenta Women's Weaving Cooperative (14 members). We seek
GHS 30,000 to buy a bulk order of yarn directly from the factory, cutting out middlemen and
raising our margins from 15% to about 35%. The cooperative has operated for 6 years and
holds GHS 9,000 in our group account. We propose repayment of GHS 2,000 monthly over
16 months, backed by our group savings and joint liability agreement.""",

    "L006": """Hi,
This is Kofi. I saw your advert. I want GHS 50,000 to start a car washing business, a
provision shop, and also import phones from Dubai. I am 22 and full of energy. I have not
started any of these yet but my friends say I am very business minded. I will pay back in
one year when the businesses are booming. No collateral but I am trustworthy."""
}

# Gold-standard labels for three letters (for Section 4 evaluation)
GOLD = {
    "L001": {
        "applicant_name": "Akosua Mensah",
        "amount_ghs": 8000,
        "purpose": "buy deep freezer / expand into frozen foods",
        "monthly_profit_ghs": 900,
        "has_collateral_or_guarantor": True,
        "repayment_months": 20
    },
    "L003": {
        "applicant_name": "Efua Darko",
        "amount_ghs": 15000,
        "purpose": "industrial sewing machines and fabric stock",
        "monthly_profit_ghs": 2800,
        "has_collateral_or_guarantor": True,
        "repayment_months": 15
    },
    "L006": {
        "applicant_name": "Kofi",
        "amount_ghs": 50000,
        "purpose": "car wash, provision shop, phone imports",
        "monthly_profit_ghs": None,
        "has_collateral_or_guarantor": False,
        "repayment_months": 12
    }
}

print(f"{len(LETTERS)} letters loaded.")

6 letters loaded.


Section 3 — Prompt Engineering for the Decision Support System

Part 3.1 — Component 1: Summarization

In [7]:
# TODO: Write SUMMARY_PROMPT_V1 — your first, naive attempt (e.g. just "Summarize this:").
#   Run it on L002 and L006. Read the output critically.



# TODO: Now write SUMMARY_PROMPT_V2 as a proper template with:
#   - a system prompt giving the LLM a ROLE (e.g. "You are an assistant to a microfinance
#     loan officer...") and constraints (factual, neutral, no invented details, 3-4 sentences)
#   - a user prompt template like: f"Summarize this loan application:\n\n{letter_text}"
#   Run V2 on the same two letters at temperature=0.

# TODO: Compare V1 vs V2 outputs side by side. Keep both prompt versions in this notebook.

# 1. Define prompts
SUMMARY_PROMPT_V1 = "Summarize this:"

SYSTEM_PROMPT_V2 = (
    "You are an assistant to a microfinance loan officer. "
    "Summarize loan applications in 3 to 4 factual, neutral sentences. "
    "Include applicant details, loan amount, repayment, and collateral. "
    "Do not invent details."
)


# 2. Run and print side-by-side comparison
for lid in ["L002", "L006"]:
    letter = LETTERS[lid]

    # Run V1
    v1_res = ask_llm(f"{SUMMARY_PROMPT_V1}\n\n{letter}", temperature=0.7)
    v1_text = v1_res[0] if isinstance(v1_res, tuple) else v1_res

    # Run V2
    v2_res = ask_llm(
        f"Summarize this application:\n\n{letter}",
        system_prompt=SYSTEM_PROMPT_V2,
        temperature=0.0,
    )
    v2_text = v2_res[0] if isinstance(v2_res, tuple) else v2_res

    print(f"=== {lid} ===")
    print(f"V1 Output:\n{v1_text}\n")
    print(f"V2 Output:\n{v2_text}\n")


=== L002 ===
V1 Output:
Kwame Boateng, a commercial driver in Kumasi, is seeking a loan of GHS 25,000 to repair his vehicle's engine and pay off personal debts. He's experiencing a slow period in business, but expects it to improve after the festive season. He doesn't have collateral, but promises to repay the loan as soon as possible.

V2 Output:
Kwame Boateng, a commercial driver from Kumasi, has applied for a loan of GHS 25,000. He intends to use the funds to repair his trotro engine and settle personal debts. The applicant has not specified a repayment plan or provided collateral for the loan. He expresses confidence that his business will improve after the festive season, implying that he will repay the loan at that time.

=== L006 ===
V1 Output:
Kofi, a 22-year-old, is requesting GHS 50,000 to start three businesses: a car washing service, a provision shop, and a phone import business from Dubai. He has no prior experience, but claims to be "business-minded" based on his friends'

1. V1 permitted subjective, unverified claims and informal phrasing from the applicants. Example of the subjective claim: in L002, V1 says Kwame "expects it to inprove after the festive season" and "promises to repay the loan as soon as possible." V2 reframes this as: "the applicant has not specified a repayment plan, which implies that he will repay the loanat that time."
For the Informal tone: In L006, V1 includes converstional quotes like "claims to be business-minded' based on his friends' opinions" and "offers his trustworthiness as collateral." V2 provides a clear, professional fact-check: "No collateral has been offered, claims to be trustworthy, but has not yet established any of the proposed businesses."
V2 solved this issues by enforcing a strict 3-4 sentence limit, removing unneccessary informal information.

2. In loan evaluations, loan officers use summaries to decide if a borrower is a safe choice. If the AI makes up details like fake payment dates, fake collateral, or fake income, the officer might make the wrong choice. They could approve a risky loan or reject a good applicant based on lies.

  This is called Hallucination

Part 3.2 — Component 2: Structured extraction (JSON)